# Phase 1b — Line Performance Analysis (Best Solution Plan)

## Objective
Measure each team’s “Line 1 vs Line 2” offensive gap using expected goals (xG), adjusted for:
- time on ice (TOI) so more minutes ≠ automatically better
- opponent defensive matchups (tougher opponents lower raw production)

Final metric per team:
- Disparity Ratio = (Adjusted Line 1 offense) / (Adjusted Line 2 offense)
Rank teams from largest ratio to smallest.

---

## Key idea (best approach)
Use record/segment-level rows (record_id) so we keep matchup info (opponent team / D-pair / goalie) and TOI.

We will estimate each line’s “true offensive xG rate” with a rate model:
- Outcome: xG_for in each segment
- Exposure: TOI (minutes) in that segment
- Controls: opponent defensive strength + home/away

This produces a matchup-adjusted xG per 60 minutes (xG60) for each line.

---

## Step 1 — Build a long-format offense table (record-level; no aggregation)
Create TWO rows per record:

Home offense row:
- team = home_team
- line = home_off_line
- xg_for = home_xg
- opp_team = away_team
- opp_def_pair = away_def_pairing
- opp_goalie = away_goalie
- is_home = 1
- toi = toi

Away offense row:
- team = away_team
- line = away_off_line
- xg_for = away_xg
- opp_team = home_team
- opp_def_pair = home_def_pairing
- opp_goalie = home_goalie
- is_home = 0
- toi = toi

Convert TOI to minutes if TOI is stored in seconds.

---

## Step 2 — Label Line 1 vs Line 2 (light aggregation for labeling only)
Aggregate total TOI by (team, line):
- Line 1 = highest TOI line for that team
- Line 2 = second-highest TOI line for that team

Stability rule:
- require Line 2 TOI >= ~100 minutes (or similar threshold) to compute a ratio.

---

## Step 3 — Fit a matchup-adjusted xG rate model (TOI offset)
Fit a Poisson rate model (GLM) where:
- xg_for is predicted from:
  - line (main effect)
  - opponent team
  - opponent defensive pairing
  - opponent goalie
  - is_home
- TOI enters as an “offset” using log(toi_minutes)

Interpretation:
- the line effect estimates offensive quality after adjusting for matchup difficulty and TOI.

Stability improvement:
- collapse rare line / D-pair / goalie labels into "OTHER".

---

## Step 4 — Convert model output into adjusted xG60 per line
For each (team, line):
- predict expected xG for 60 minutes under neutral context:
  - opponent/team/goalie = "OTHER" (or baseline)
  - is_home = 0 (neutral)

The prediction is the line’s adjusted xG60.

---

## Step 5 — Compute disparity ratio and rank teams
For each team:
- find adjusted_xG60(Line 1) and adjusted_xG60(Line 2)
- Disparity Ratio = Line1 / Line2

Rank teams: highest ratio = largest line quality disparity.

---

## Required quality checks
- Confirm TOI units (seconds vs minutes)
- Ensure Line 2 has enough TOI so ratios aren’t noisy
- Sanity-check top/bottom teams by inspecting their Line1/Line2 TOI and raw xG60

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("/content/drive/MyDrive/Wharton Data Science Competition 2026/whl_2025.csv")

In [ ]:
df.head().T

,0,1,2,3,4
game_id,game_1,game_1,game_1,game_1,game_1
record_id,record_1,record_2,record_3,record_4,record_5
home_team,thailand,thailand,thailand,thailand,thailand
away_team,pakistan,pakistan,pakistan,pakistan,pakistan
went_ot,0,0,0,0,0
home_off_line,PP_kill_dwn,second_off,first_off,second_off,second_off
home_def_pairing,PP_kill_dwn,second_def,second_def,first_def,second_def
away_off_line,PP_up,second_off,second_off,second_off,first_off
away_def_pairing,PP_up,second_def,second_def,first_def,second_def
home_goalie,player_id_142,player_id_142,player_id_142,player_id_142,player_id_142


In [ ]:
def make_off_table_format(df_in: pd.DataFrame) -> pd.DataFrame:
  home = pd.DataFrame({
      "record_id": df_in["record_id"].values,
      "game_id": df_in["game_id"].values,
      "team": df_in["home_team"].values,
      "opp_team": df_in["away_team"].values,
      "is_home": 1,
      "off_line": df_in["home_off_line"].values,
      "def_pairing": df_in["away_def_pairing"].values,
      "opp_goalie": df_in["away_goalie"].values,
      "toi": df_in["toi"].values,
      "xg_for": df_in["home_xg"].values,
      "xg_against": df_in["away_xg"].values
  })

  away = pd.DataFrame({
      "record_id": df_in["record_id"].values,
      "game_id": df_in["game_id"].values,
      "team": df_in["away_team"].values,
      "opp_team": df_in["home_team"].values,
      "is_home": 0,
      "off_line": df_in["away_off_line"].values,
      "def_pairing": df_in["home_def_pairing"].values,
      "opp_goalie": df_in["home_goalie"].values,
      "toi": df_in["toi"].values,
      "xg_for": df_in["away_xg"].values,
      "xg_against": df_in["home_xg"].values
  })

  return pd.concat([home, away], ignore_index=True)

In [ ]:
combo = make_off_table_format(df)
combo

,record_id,game_id,team,opp_team,is_home,off_line,def_pairing,opp_goalie,toi,xg_for,xg_against
0,record_1,game_1,thailand,pakistan,1,PP_kill_dwn,PP_up,player_id_106,628.80,0.1754,1.4645
1,record_2,game_1,thailand,pakistan,1,second_off,second_def,player_id_106,197.61,0.0000,0.0928
2,record_3,game_1,thailand,pakistan,1,first_off,second_def,player_id_106,47.06,0.0000,0.1880
3,record_4,game_1,thailand,pakistan,1,second_off,first_def,player_id_106,44.60,0.1211,0.0727
4,record_5,game_1,thailand,pakistan,1,second_off,second_def,player_id_106,274.65,0.1207,0.0769
...,...,...,...,...,...,...,...,...,...,...,...
51649,record_25823,game_999,thailand,oman,0,second_off,first_def,player_id_266,138.33,0.1212,0.0909
51650,record_25824,game_999,thailand,oman,0,second_off,first_def,player_id_266,189.08,0.0931,0.0000
51651,record_25825,game_999,thailand,oman,0,first_off,first_def,player_id_266,155.99,0.0000,0.2171
51652,record_25826,game_999,thailand,oman,0,second_off,first_def,player_id_266,218.54,0.0000,0.1820


In [ ]:
# cleaning up combo dataframe
def create_combo(combo: pd.DataFrame) -> pd.DataFrame:
  combo = combo[combo["toi"]>0].copy()

  # total TOI per team-line
  line_totals = (
      combo.groupby(["team", "off_line"], as_index=False).agg(
          line_toi_sec=("toi", "sum"),
          line_xg = ("xg_for", "sum"),
          n_records=("record_id", "nunique")
      )
  )

  line_totals["line_toi_min"] = line_totals["line_toi_sec"] / 60.0 # convert to minutes

  # rank within team by TOI (line 1 = highest TOI, line 2 = second)
  line_totals["line_rank"] = line_totals.groupby("team")["line_toi_min"].rank(method="first", ascending=False)

  line12 = line_totals[line_totals["line_rank"].isin([1,2])].copy() # boolean mask! :)

  # stability filter: require Line 2 has enough minutes
  MIN_LINE2_TOI_MIN = 100.0
  teams_ok = line12[(line12["line_rank"] == 2) & (line12["line_toi_min"] >= MIN_LINE2_TOI_MIN)]["team"].unique()
  line12 = line12[line12["team"].isin(teams_ok)].copy()

  print("Teams retained: ", len(teams_ok))
  return line12



In [ ]:
combo = make_off_table_format(df_in = df)
combo.head()

,record_id,game_id,team,opp_team,is_home,off_line,def_pairing,opp_goalie,toi,xg_for,xg_against
0,record_1,game_1,thailand,pakistan,1,PP_kill_dwn,PP_up,player_id_106,628.80,0.1754,1.4645
1,record_2,game_1,thailand,pakistan,1,second_off,second_def,player_id_106,197.61,0.0000,0.0928
2,record_3,game_1,thailand,pakistan,1,first_off,second_def,player_id_106,47.06,0.0000,0.1880
3,record_4,game_1,thailand,pakistan,1,second_off,first_def,player_id_106,44.60,0.1211,0.0727
4,record_5,game_1,thailand,pakistan,1,second_off,second_def,player_id_106,274.65,0.1207,0.0769


In [ ]:
agg_12 = create_combo(combo)
agg_12

Teams retained:  32


,team,off_line,line_toi_sec,line_xg,n_records,line_toi_min,line_rank
3,brazil,first_off,108743.00,81.0273,708,1812.383333,1.0
4,brazil,second_off,107499.01,81.7372,709,1791.650167,2.0
8,canada,first_off,112738.79,74.0168,692,1878.979833,1.0
9,canada,second_off,108486.72,70.4763,683,1808.112000,2.0
13,china,first_off,114861.38,79.6366,677,1914.356333,1.0
...,...,...,...,...,...,...,...
149,uk,second_off,112425.66,72.5518,684,1873.761000,2.0
153,usa,first_off,107480.47,81.2704,675,1791.341167,1.0
154,usa,second_off,104909.22,58.1439,679,1748.487000,2.0
158,vietnam,first_off,110912.34,61.7860,689,1848.539000,1.0


In [ ]:
agg_12["xg60"] = 60 * agg_12["line_xg"] / agg_12["line_toi_min"]
agg_12

,team,off_line,line_toi_sec,line_xg,n_records,line_toi_min,line_rank,xg60
3,brazil,first_off,108743.00,81.0273,708,1812.383333,1.0,2.682456
4,brazil,second_off,107499.01,81.7372,709,1791.650167,2.0,2.737271
8,canada,first_off,112738.79,74.0168,692,1878.979833,1.0,2.363521
9,canada,second_off,108486.72,70.4763,683,1808.112000,2.0,2.338670
13,china,first_off,114861.38,79.6366,677,1914.356333,1.0,2.495980
...,...,...,...,...,...,...,...,...
149,uk,second_off,112425.66,72.5518,684,1873.761000,2.0,2.323193
153,usa,first_off,107480.47,81.2704,675,1791.341167,1.0,2.722108
154,usa,second_off,104909.22,58.1439,679,1748.487000,2.0,1.995230
158,vietnam,first_off,110912.34,61.7860,689,1848.539000,1.0,2.005454


In [ ]:
wide = agg_12.pivot(index="team", columns="off_line", values="xg60").reset_index()
wide["disparity_ratio"] = wide["first_off"]/wide["second_off"]
wide.sort_values("disparity_ratio", ascending=False)

off_line,team,first_off,second_off,disparity_ratio
30,usa,2.722108,1.995230,1.364308
22,saudi_arabia,2.230361,1.636620,1.362785
6,guatemala,2.787262,2.056095,1.355610
28,uae,1.982444,1.462816,1.355225
4,france,2.542702,1.897530,1.340006
7,iceland,2.574164,1.944120,1.324077
24,singapore,2.614200,2.083017,1.255006
15,new_zealand,2.417589,1.973352,1.225118
18,panama,2.531816,2.111680,1.198958
19,peru,2.372764,1.980904,1.197819


In [ ]:
# NOT USING THIS STATISTICAL MODEL
# import statsmodels.formula.api as smf
# import statsmodels.api as sm

# # Rebuild combo
# combo = make_off_table_format(df)
# combo = combo[combo["toi"] > 0].copy()
# combo["toi_min"] = combo["toi"]/60.0

# def cap_rare(s: pd.Series, min_count=200, other="OTHER") -> pd.Series:
#     vc = s.value_counts(dropna=False)
#     keep = vc[vc >= min_count].index
#     return s.where(s.isin(keep), other)

# # Fit Poisson GLM rate model
# # All the teams have enough info for the model
# model_df = combo.copy()

# # cast to string to clean/normalize categories and replace rare levels with "OTHER"
# # makes the processes much safer when column is consistent string dtype
# model_df["off_line"]    = cap_rare(model_df["off_line"].astype(str),    min_count=200)
# model_df["def_pairing"] = cap_rare(model_df["def_pairing"].astype(str), min_count=200)
# model_df["opp_goalie"]  = cap_rare(model_df["opp_goalie"].astype(str),  min_count=200)
# model_df["opp_team"]    = model_df["opp_team"].astype(str)

# formula = "xg_for ~ C(team) + C(off_line) + C(opp_team) + C(def_pairing) + C(opp_goalie) + is_home"

# glm = smf.glm(
#     formula=formula,
#     data=model_df,
#     family=sm.families.Poisson(),
#     offset=np.log(model_df["toi_min"])
# ).fit()

# print(glm.summary())

## Upgrade Matchup Adjusted
We add matchup-adusted line strength and we shrink the data to make it more stable.

In [ ]:
def phase1b_score(combo: pd.DataFrame,
                  min_line_2_toi_min: float = 100.0,
                  use_shrinkage: bool = False,
                  prior_toi_sec: float = 600.0,
                  weight_mode: str = "team"  # "team" or "league"
                  ) -> pd.DataFrame:
  """
  Returns a ranked table with a matchup-adjusted line strength and disparity ratio.

  combo must contain: team, off_line, def_pairing, toi (seconds), xg_for
  (record_id optional)

  weight_mode:
    - "team": weight matchups by each team's own TOI mix (reflects deployment)
    - "league": weight matchups by the league TOI mix for that line (standardized)
  """

  # --- Phase 1b scope (restrict to the two lines and two defensive pairings) ---
  cb = combo[
      (combo["toi"] > 0) &  # only rows with positive TOI
      (combo["off_line"].isin(["first_off", "second_off"])) &  # keep only first/second offensive lines
      (combo["def_pairing"].isin(["first_def", "second_def"]))  # keep only first/second defensive pairings
  ].copy()

  # --- Stability filter: require enough second-line minutes for each team ---
  # Compute TOI per team-line; drop teams whose second line barely played.
  toi_line = cb.groupby(["team", "off_line"], as_index=False)["toi"].sum()
  toi_line["toi_min"] = toi_line["toi"] / 60.0

  teams_ok = toi_line[
      (toi_line["off_line"] == "second_off") &
      (toi_line["toi_min"] >= min_line_2_toi_min)
  ]["team"].unique()

  cb = cb[cb["team"].isin(teams_ok)].copy()

  # --- Aggregate to the matchup-cell level: team × line × def_pairing ---
  # This preserves the "who did the line face?" information needed for adjustment.
  tl_d = (cb.groupby(["team", "off_line", "def_pairing"], as_index=False).agg(
      toi_sec=("toi", "sum"),
      xg=("xg_for", "sum")
  ))

  # --- League baselines by def_pairing (difficulty of first_def vs second_def) ---
  # League average xG rates differ vs first_def vs second_def; we normalize by these.
  league_vs = (cb.groupby("def_pairing", as_index=False).agg(
      toi_sec=("toi", "sum"),
      xg=("xg_for", "sum")
  ))
  league_vs["mu_xg60_vs_def"] = 3600.0 * league_vs["xg"] / league_vs["toi_sec"]      # baseline xG60 vs each def type
  league_vs["mu_xg_per_sec_vs_def"] = league_vs["xg"] / league_vs["toi_sec"]         # baseline xG/sec vs each def type (for shrinkage)

  # Attach the appropriate league baseline to each matchup cell in tl_d (by def_pairing).
  tl_d = tl_d.merge(
      league_vs[["def_pairing", "mu_xg60_vs_def", "mu_xg_per_sec_vs_def"]],
      on="def_pairing", how="left"
  )

  # --- Compute matchup-cell scoring rate (optionally shrink toward league baseline) ---
  # Shrinkage stabilizes small-TOI matchup cells by pulling them toward the league rate vs that def_pairing.
  if use_shrinkage:
    tl_d["xg_shrunk"] = tl_d["xg"] + prior_toi_sec * tl_d["mu_xg_per_sec_vs_def"]
    tl_d["toi_shrunk"] = tl_d["toi_sec"] + prior_toi_sec
    tl_d["xg60_used"] = 3600.0 * tl_d["xg_shrunk"] / tl_d["toi_shrunk"]
  else:
    tl_d["xg60_used"] = 3600.0 * tl_d["xg"] / tl_d["toi_sec"]

  # --- Normalize for defensive matchup difficulty (first_def vs second_def) ---
  # rel_index > 1 means "better than league average vs that def_pairing"
  tl_d["rel_index"] = tl_d["xg60_used"] / tl_d["mu_xg60_vs_def"]

  # --- Compute weights for combining matchup cells into one line score ---
  # Team weights reflect each team's actual matchup mix (deployment):
  tl_d["w_team"] = tl_d["toi_sec"] / tl_d.groupby(["team", "off_line"])["toi_sec"].transform("sum")

  # League-mixture weights standardize away deployment:
  # For each line, compute the league-wide TOI share vs each def_pairing.
  league_mix = (tl_d.groupby(["off_line", "def_pairing"], as_index=False)["toi_sec"].sum())
  league_mix["w_league"] = league_mix["toi_sec"] / league_mix.groupby("off_line")["toi_sec"].transform("sum")

  # Attach league mixture weights to each row (by off_line + def_pairing).
  tl_d = tl_d.merge(
      league_mix[["off_line", "def_pairing", "w_league"]],
      on=["off_line", "def_pairing"],
      how="left"
  )

  # Choose which weights to use for the final line score.
  if weight_mode not in ["team", "league"]:
    raise ValueError("weight_mode must be 'team' or 'league'")
  weight_col = "w_team" if weight_mode == "team" else "w_league"

  # --- Line strength: weighted average of rel_index across def_pairings ---
  # This collapses (vs first_def, vs second_def) back into one number per line.
  line_strength = (tl_d.assign(weighted=tl_d[weight_col] * tl_d["rel_index"])
                     .groupby(["team", "off_line"], as_index=False)["weighted"].sum()
                     .rename(columns={"weighted": "line_strength"}))

  # --- Disparity ratio: first line strength divided by second line strength ---
  wide = line_strength.pivot(index="team", columns="off_line", values="line_strength").reset_index()
  wide["disparity_ratio"] = wide["first_off"] / wide["second_off"]
  wide = wide.sort_values("disparity_ratio", ascending=False).reset_index(drop=True)

  return wide

In [ ]:
# baseline-adjusted (no shrinkage)
ranked = phase1b_score(combo, min_line_2_toi_min=500.0, use_shrinkage=False)
top10 = ranked

# optionally: more stable (shrinkage) team
ranked_shrunk_team = phase1b_score(combo, min_line_2_toi_min=100.0, use_shrinkage=True, prior_toi_sec=600.0, weight_mode="team")
top10_shrunk_team = ranked_shrunk_team

# optionally: more stable (shrinkage) league
ranked_shrunk_league = phase1b_score(combo, min_line_2_toi_min=100.0, use_shrinkage=True, prior_toi_sec=600.0, weight_mode="league")
top10_shrunk_league = ranked_shrunk_league

In [ ]:
top10

off_line,team,first_off,second_off,disparity_ratio
0,usa,1.208151,0.882185,1.369498
1,guatemala,1.248700,0.916365,1.362666
2,saudi_arabia,0.993204,0.732023,1.356793
3,uae,0.884281,0.653193,1.353783
4,france,1.132607,0.841387,1.346118
5,iceland,1.146020,0.867832,1.320555
6,singapore,1.166361,0.930091,1.254030
7,new_zealand,1.080331,0.874148,1.235868
8,peru,1.058144,0.877459,1.205919
9,panama,1.126417,0.938064,1.200789


In [ ]:
top10_shrunk_team

off_line,team,first_off,second_off,disparity_ratio
0,usa,1.205849,0.883555,1.364769
1,guatemala,1.245976,0.917284,1.358333
2,saudi_arabia,0.993255,0.734887,1.351576
3,uae,0.885498,0.656895,1.348006
4,france,1.131184,0.843113,1.341675
5,iceland,1.144418,0.869322,1.316450
6,singapore,1.164558,0.930851,1.251069
7,new_zealand,1.079437,0.875572,1.232836
8,peru,1.057523,0.878751,1.203438
9,panama,1.125063,0.938738,1.198484


In [ ]:
# top10.to_csv("/content/drive/MyDrive/Wharton Data Science Competition 2026/disp_ratio_no_shrinkage.csv")

In [ ]:
# top10_shrunk_league.to_csv("/content/drive/MyDrive/Wharton Data Science Competition 2026/disp_ratio_league.csv")

In [ ]:
# top10_shrunk_team.to_csv("/content/drive/MyDrive/Wharton Data Science Competition 2026/disp_ratio_team.csv")

In [ ]:
top10_shrunk_league

off_line,team,first_off,second_off,disparity_ratio
0,usa,1.205369,0.881931,1.366739
1,guatemala,1.246046,0.917441,1.358175
2,saudi_arabia,0.993424,0.734918,1.351748
3,uae,0.885522,0.656939,1.347951
4,france,1.131180,0.843036,1.341793
5,iceland,1.144196,0.869498,1.315927
6,singapore,1.164565,0.930507,1.251538
7,new_zealand,1.079466,0.875488,1.232988
8,peru,1.058088,0.878702,1.204148
9,panama,1.125239,0.938846,1.198534


In [ ]:
# The disparity ratios rankings are the same regardless of shrinkage and what type of shrinkage
(top10["team"] == top10_shrunk_league["team"]) == (top10["team"] == top10_shrunk_team["team"])

,team
0,True
1,True
2,True
3,True
4,True
5,True
6,True
7,True
8,True
9,True
